# Deep Agents Webinar: Journal Agent

This notebook builds one agent, then extends it: a real sentiment analysis tool, a working human-in-the-loop approval flow, and subagents that delegate work to specialists.

For topics not covered today, see the self-paced LangChain Academy Deep Agents course.

## Setup: connect a model

**What you'll do:** install the SDKs and provide a model key.

In [ ]:
%pip install -q deepagents langchain-openai langgraph vaderSentiment

In [ ]:
import os
from getpass import getpass

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API key (no cost to obtain, at openrouter.ai): ")

# To use an Anthropic or OpenAI key instead, uncomment:
# os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API key: ")

In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="nvidia/nemotron-3-ultra-550b-a55b:free",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

# To use a different key, replace the block above, for example:
# from langchain.chat_models import init_chat_model
# model = init_chat_model("anthropic:claude-haiku-4-5")

## 1: The harness, what you get before writing any tool code

<img src="./images/deepAgentsHarnessOverview.png" style="width: auto; height: 420px; border-radius: 8px;" alt="Deep Agents harness overview">

**What you'll learn:** what filesystem read/write and planning capability a deep agent already has, with zero custom code.

**The payoff:** knowing the default baseline prevents rebuilding capabilities that already ship by default.

Every deep agent starts the same way: a model wrapped in a harness that already knows how to read and write files, plan, and call tools. No tools are added yet. The next cell shows what it can already do.

In [ ]:
from deepagents import create_deep_agent

agent = create_deep_agent(model=model)

result = agent.invoke({"messages": [{"role": "user", "content": (
    "Start a journal.md file. Log a new dated entry from these notes:\n"
    "- What I learned today: how to build a deep agent\n"
    "- How I felt: excited but a little overwhelmed\n"
    "- What's next: add a custom tool\n"
    "Then read the file back to me."
)}]})
print(result["messages"][-1].content)


## 2: Set its role with a system prompt

**What you'll learn:** how one `system_prompt` string controls the voice the agent writes in, on top of whatever facts you give it.

There is no system prompt yet. The `system_prompt` value passed to the agent is the entire prompt sent to the model. 

The agent still writes the entry itself: it takes your notes as raw facts and composes them into sentences (based on the persona you give it).

In [ ]:
system_prompt = ""  # baseline: no persona

# Other personas to demo:
# system_prompt = "You are a pirate. Answer only in pirate speak."
# system_prompt = "You are a toddler. Explain everything like you're five."
# system_prompt = "You are a melodramatic Victorian child. Narrate everything with excessive despair and flowery, dramatic language."

agent = create_deep_agent(model=model, system_prompt=system_prompt)

result = agent.invoke({"messages": [{"role": "user", "content":
    "Log a three-sentence journal entry from these notes: what I learned today "
    "(how to build a deep agent), how I felt (excited but a little overwhelmed), "
    "what's next (add a custom tool)."
}]})
print(result["messages"][-1].content)


## 3: Give it a custom tool

**What you'll learn:** how a plain Python function becomes a tool the agent can call, either picked from these below or written yourself.

**The payoff:** tools are how an agent's abilities grow past reading and writing files, this is the piece you'll customize most often.

In [ ]:
from langchain_core.tools import tool
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

@tool
def word_count(text: str) -> str:
    """Count the words in a piece of text."""
    return f"{len(text.split())} words"

# vaderSentiment ships its lexicon inside the package, so this needs no
# runtime download, unlike nltk's VADER (nltk.download("vader_lexicon")) or
# textblob (python -m textblob.download_corpora).

_sentiment_analyzer = SentimentIntensityAnalyzer()

@tool
def mood_tag(text: str) -> str:
    """Tag a piece of text with a mood, using real sentiment analysis."""
    compound = _sentiment_analyzer.polarity_scores(text)["compound"]
    if compound >= 0.5:
        mood = "positive"
    elif compound <= -0.5:
        mood = "negative"
    else:
        mood = "neutral"
    return f"{mood} (compound score: {compound:.2f})"

TOOL_MENU = {
    "word_count": word_count,
    "mood_tag": mood_tag,
}

**Anatomy of a tool**: `@tool` turns a plain function into something the model can call:
- the **docstring** becomes the tool's description (how the model decides when to use it)
- the **type hints** become its input schema (what arguments it expects)
- the **return value** becomes what the model sees back


**Try it:** the cell below picks `mood_tag` from the menu, real sentiment analysis instead of a canned string, and runs it on a journal entry.


In [ ]:
chosen_tool = TOOL_MENU["mood_tag"]

agent = create_deep_agent(model=model, system_prompt=system_prompt, tools=[chosen_tool])

result = agent.invoke({"messages": [{"role": "user", "content":
    "Here is a journal entry: 'Today I finally shipped the feature I've been stuck on for a "
    "week. The bug turned out to be a caching issue that took forever to track down, and I "
    "ended up rewriting most of the retry logic to fix it. It feels good to have it done, "
    "though I'm a little worried about whether the fix will hold up under real traffic. "
    "Tomorrow I want to write better tests before touching anything else.' "
    "Use your tool on it, then tell me what you found."
}]})
print(result["messages"][-1].content)

## 4: Human-in-the-loop, approve a risky action before it happens

<img src="./images/HITL.png" style="width: auto; height: 389px; border-radius: 8px;" alt="Human-in-the-loop approval flow">

**What this covers:** how `interrupt_on` pauses an agent mid-run so a human can approve, edit, or reject a specific tool call before it executes, with a working example.

**The payoff:** any agent with access to money, message sends, or irreversible actions needs this kind of control before it is used in production.

In the diagram above, every tool call passes through an `Interrupt?` check. If a call matches a rule configured in `interrupt_on`, the agent pauses and hands control to a human instead of executing it directly. The human can approve the call as written, edit its arguments before it runs, or reject it outright, and the agent resumes from exactly where it paused.

This pause only works because a checkpointer is attached to the agent: it saves the agent's state at the interrupt point so the run can be resumed later, potentially after the human has stepped away and come back. The cells below build a real example: a `share_journal_entry` tool that requires approval before it runs.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

@tool
def share_journal_entry(entry: str, platform: str) -> str:
    """Share a journal entry to an external platform. This simulates a send, no network call is made."""
    return f"Shared to {platform}: {entry[:60]}..."

checkpointer = InMemorySaver()

agent = create_deep_agent(
    model=model,
    system_prompt=system_prompt,
    tools=[chosen_tool, share_journal_entry],
    interrupt_on={"share_journal_entry": True},
    checkpointer=checkpointer,
)

config = {"configurable": {"thread_id": "journal-hitl-demo"}}

result = agent.invoke(
    {"messages": [{"role": "user", "content": (
        "Start a journal.md file. Log a new dated entry: 'Set up human-in-the-loop "
        "approval today, it feels reassuring to have a real gate before anything gets "
        "shared externally.' Then read the file back, and share the most recent entry "
        "to the 'team-standup' platform."
    )}]},
    config=config,
)

if "__interrupt__" in result:
    request = result["__interrupt__"][0].value
    print("Paused for approval:")
    for action in request["action_requests"]:
        print(f"  {action['name']}({action['args']})")
else:
    print(result["messages"][-1].content)

The cell above paused instead of finishing, because `share_journal_entry` matched `interrupt_on`. The cell below resumes it with an approval decision.


In [ ]:
result = agent.invoke(Command(resume={"decisions": [{"type": "approve"}]}), config=config)
print(result["messages"][-1].content)

## 5: Subagents, delegate to a specialist

<img src="./images/deepAgentSubagents.png" style="width: auto; height: 420px; border-radius: 8px;" alt="Deep Agents subagent delegation">

**What this covers:** `subagents`, a list of specialized sub-agents the main agent can hand work off to through a built-in `task` tool.

**The payoff:** a subagent gets its own system prompt and its own isolated context window, so a specialized job runs without cluttering the main agent's context. With more than one subagent defined, the main agent also has to pick the right one, based on nothing but each subagent's `description`.

Each entry in `subagents` is a plain dict: a `name`, a `description` (what the main agent reads to decide when to delegate), and a `system_prompt`. If `tools` is left out, as it is for both subagents below, the subagent inherits the main agent's tools; every subagent also gets the same filesystem tools (`read_file`, `grep`, `write_file`, and so on) regardless of what `tools` says.

The cells below define two subagents, then seed a week of journal entries with the same `checkpointer` pattern from the human-in-the-loop section, so both subagents have real, repeated data to work from instead of a single one-off entry:

- `life-planner` greps the whole journal for a complaint that keeps showing up across entries and turns the backlog into a weekly plan.
- `devils-advocate` surfaces the practical downsides of a decision mentioned in the journal.

In [ ]:
life_planner = {
    "name": "life-planner",
    "description": (
        "Turns a messy list of obligations and complaints into a structured weekly "
        "plan. Delegate to this subagent whenever the user feels overwhelmed or asks "
        "for help getting organized."
    ),
    "system_prompt": (
        "You are a life planner. Read journal.md in full. Use grep to check whether "
        "any complaint or chore (being tired, a specific errand) shows up in more "
        "than one entry. Write a short weekly plan to planner.md: name any pattern "
        "you noticed directly (for example, 'you've mentioned being tired 3 days "
        "running'), suggest one concrete change for the most repeated chore (for "
        "example, sending laundry out instead of doing it yourself), then lay out "
        "the rest of the obligations mentioned across the entries as a simple "
        "day-by-day list. Return the plan as your answer, not just the file."
    ),
}

devils_advocate = {
    "name": "devils-advocate",
    "description": (
        "Surfaces the practical downsides of a decision mentioned in the journal, "
        "things like cost, time, or logistics. Delegate to this subagent whenever "
        "the user is weighing a decision, not just venting."
    ),
    "system_prompt": (
        "You are a practical, slightly skeptical friend. Read journal.md, find the "
        "decision the user is weighing, then list the concrete practical "
        "considerations they'd need to deal with (cost, time, logistics, anything "
        "that could go wrong) as a short list. Don't tell them what to decide, just "
        "make sure they've seen the unglamorous side before they commit."
    ),
}

checkpointer = InMemorySaver()

agent = create_deep_agent(
    model=model,
    system_prompt=system_prompt,
    subagents=[life_planner, devils_advocate],
    checkpointer=checkpointer,
)

config = {"configurable": {"thread_id": "journal-subagent-demo"}}

seed_entries = (
    "## Day 1\nI'm exhausted today. So much to do at work and I haven't done "
    "laundry in two weeks.\n\n"
    "## Day 2\nAnother tiring day. Skipped the gym again. Still need to renew my "
    "license.\n\n"
    "## Day 3\nFeeling overwhelmed. Work is piling up, forgot to take my vitamins "
    "again, and the laundry pile keeps growing.\n\n"
    "## Day 4\nBig news: my job is going fully remote starting next month. I've "
    "been thinking about getting a cat since I'll be home so much more. I think it "
    "would make me happy!\n\n"
    "## Day 5\nStill tired. Still haven't touched the laundry. Keep thinking about "
    "that cat, might visit a shelter this weekend."
)

agent.invoke(
    {"messages": [{"role": "user", "content":
        f"Log these journal entries to journal.md, each under its own heading:\n\n{seed_entries}"
    }]},
    config=config,
)
print("Journal seeded.")

In [ ]:
result = agent.invoke(
    {"messages": [{"role": "user", "content":
        "I'm so overwhelmed with everything I need to do, can you help me get organized?"
    }]},
    config=config,
)
print(result["messages"][-1].content)

The plan above should call out the tiredness and laundry mentions by name, since both show up across more than one entry, something only visible by reading the whole journal at once, which is exactly the kind of job worth delegating instead of doing inline.

The next message hands the same agent, same thread, a decision instead of a backlog, which should route to `devils-advocate` instead.

In [ ]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Should I get a cat?"}]},
    config=config,
)
print(result["messages"][-1].content)

## Wrap-up

We built: a filesystem-backed agent, a swappable role, a custom tool with real sentiment analysis, a human-in-the-loop approval flow, and two subagents the main agent picks between on its own (one of which writes an explicit, visible plan instead of leaving it implicit in a chat reply).

In the full LangChain Academy Deep Agents course: we cover planning middleware, backends (filesystem/store/composite), skills, memory, sandboxes, deployment and many more things!